In [1]:
no_of_threads = "12"

import os
os.environ["VECLIB_MAXIMUM_THREADS"] = no_of_threads
os.environ["OMP_NUM_THREADS"] = no_of_threads


In [2]:


import torch
from torch import linalg as LA


import time
import statistics
from typing import Tuple
from typing_extensions import Annotated
import transformers
import lm_eval
from lm_eval.models.huggingface import HFLM
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import platform
import subprocess
import time
from datasets import load_dataset

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")

In [4]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"   

model_hf = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, dtype=torch.float32, device_map="cpu"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 214.99it/s]


In [5]:
def time_counter(no_of_threads: int, model, tokenizer, prompts: list,
                 results: list, max_new_tokens: int = 128, repeats: int = 3):
    

    torch.set_num_threads(int(no_of_threads))
    
    print(torch.get_num_threads())
    time.sleep(2)
    for x in range(1):
        n_dict={}
        n_lst=[]
        
        messages = [{"role": "user", "content": prompts[x]}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer(text, return_tensors="pt").to(model.device)
        for i in range(repeats):
            start_time=time.perf_counter()
            out_put_ids=model.generate(max_new_tokens=max_new_tokens,**model_inputs,do_sample=False)
            


            end_time=time.perf_counter()
            n_lst.append(end_time-start_time)
        mean=statistics.mean(n_lst)
        stndev=statistics.stdev(n_lst)
        n_dict['thread']=torch.get_num_threads()
        n_dict['mean']=mean
        n_dict['stdev']=stndev
        results.append(n_dict)

    
    
    
    
    
  
    
    

In [6]:
def make_prompt(item):
    lines = [item["question"]]
    for label, text in zip(item["choices"]["label"], item["choices"]["text"]):
        lines.append(f"{label}. {text}")
    lines.append("Answer with the letter and briefly explain why.")
    return "\n".join(lines)

prompts = [make_prompt(arc[i]) for i in range(10)]



In [7]:
results=[]
prompts = [make_prompt(arc[i]) for i in range(10)]
with open("prompts_arc10.json", "w") as f:
    json.dump(prompts, f, indent=2)
with open("prompts_arc10.json") as f:
    prompts = json.load(f)


In [10]:
for i in range(1,13):
    time_counter(i,model_hf,tokenizer,prompts,results)

1
2
3
4
5
6
7
8
9
10
11
12


In [11]:
results

[{'thread': 1, 'mean': 9.076502750001964, 'stdev': 0.1878217403339656},
 {'thread': 2, 'mean': 8.59536758333464, 'stdev': 0.13724575711590128},
 {'thread': 3, 'mean': 8.488060736335077, 'stdev': 0.008895667961501876},
 {'thread': 4, 'mean': 8.528385333333668, 'stdev': 0.024061873882463978},
 {'thread': 5, 'mean': 8.657956583003397, 'stdev': 0.12044394601377491},
 {'thread': 6, 'mean': 8.741211972335199, 'stdev': 0.19239450446055323},
 {'thread': 7, 'mean': 8.585482847658568, 'stdev': 0.034363572374832146},
 {'thread': 8, 'mean': 8.627764375666933, 'stdev': 0.047870021950675044},
 {'thread': 9, 'mean': 8.684989208331293, 'stdev': 0.11144680163422116},
 {'thread': 10, 'mean': 8.804451638667766, 'stdev': 0.014774661467477218},
 {'thread': 11, 'mean': 8.848402389339753, 'stdev': 0.009925228437984228},
 {'thread': 12, 'mean': 9.42688281966548, 'stdev': 0.5271532909160086}]

In [12]:
print(torch.__config__.show())

PyTorch built with:
  - GCC 4.2
  - C++ Version: 202002
  - clang 21.0.0
  - OpenMP 202011
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: DEFAULT
  - Build settings: BLAS_INFO=accelerate, BUILD_TYPE=Release, COMMIT_SHA=5c4886908584029761b579af026dcfb627c84070, CUDA_FLAGS= -DLIBCUDACXX_ENABLE_SIMPLIFIED_COMPLEX_OPERATIONS -Xfatbin -compress-all  -Wno-deprecated-gpu-targets --expt-extended-lambda -DCUB_WRAPPED_NAMESPACE=at_cuda_detail -DDISABLE_CUSPARSE_DEPRECATED -DCUDA_HAS_FP16=1 -D__CUDA_NO_HALF_OPERATORS__ -D__CUDA_NO_HALF_CONVERSIONS__ -D__CUDA_NO_HALF2_OPERATORS__ -D__CUDA_NO_BFLOAT16_CONVERSIONS__ -DC10_NODEPRECATED, CXX_COMPILER=/usr/bin/c++, CXX_FLAGS= -fvisibility-inlines-hidden -DUSE_PTHREADPOOL -DNDEBUG -DUSE_KINETO -DUSE_PYTORCH_QNNPACK -DAT_BUILD_ARM_VEC256_WITH_SLEEF -DUSE_PYTORCH_METAL_EXPORT -DSYMBOLICATE_MOBILE_DEBUG_HANDLE -DUSE_COREML_DELEGATE -O2 -fPIC -DC10_NODEPRECATED -Wall -Wextra -Werror=return-type -Werror=non-vi

In [13]:
t1 = next(r['mean'] for r in results if r['thread'] == 1)

n_res = []
for r in results:
    n = r['thread']
    speedup = t1 / r['mean']
    efficiency = (speedup / n) * 100
    f = ((n / speedup - 1) / (n - 1)) * 100 if n > 1 else None   

    n_res.append({
        'threads': n,
        'mean': r['mean'],
        'speedup': speedup,
        'efficiency': efficiency,
        'serial_fraction': f,
    })

In [14]:
n_res

[{'threads': 1,
  'mean': 9.076502750001964,
  'speedup': 1.0,
  'efficiency': 100.0,
  'serial_fraction': None},
 {'threads': 2,
  'mean': 8.59536758333464,
  'speedup': 1.0559761013130124,
  'efficiency': 52.798805065650626,
  'serial_fraction': 89.398225728137},
 {'threads': 3,
  'mean': 8.488060736335077,
  'speedup': 1.0693258486179213,
  'efficiency': 35.644194953930715,
  'serial_fraction': 90.27529606047726},
 {'threads': 4,
  'mean': 8.528385333333668,
  'speedup': 1.0642697761938533,
  'efficiency': 26.60674440484633,
  'serial_fraction': 91.94818486421727},
 {'threads': 5,
  'mean': 8.657956583003397,
  'speedup': 1.0483423730514223,
  'efficiency': 20.966847461028447,
  'serial_fraction': 94.23585577883402},
 {'threads': 6,
  'mean': 8.741211972335199,
  'speedup': 1.0383574701915383,
  'efficiency': 17.30595783652564,
  'serial_fraction': 95.56713698786649},
 {'threads': 7,
  'mean': 8.585482847658568,
  'speedup': 1.0571918797179016,
  'efficiency': 15.102741138827167,
  